# Ruri v3 embedding + LogisticRegression

[cl-nagoya/ruri-v3-310m](https://huggingface.co/cl-nagoya/ruri-v3-310m) はfine-tuningせず、768次元の文章embeddingを分類に使う。`texts` と `labels` を実データに置き換える。

In [3]:
#!pip install -q -U "transformers==4.57.6" sentence-transformers scikit-learn

In [4]:
import torch
#import torch._dynamo
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

device = "cuda" if torch.cuda.is_available() else "cpu"
encoder = SentenceTransformer("cl-nagoya/ruri-v3-30m", device=device)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/7.94k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.32k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/3.78k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/1.83M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/968 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [5]:
texts = [
    "この商品には満足しています。", "とても使いやすくて便利です。",
    "期待どおりの品質でした。", "また購入したいと思います。",
    "説明が分かりやすかったです。", "価格以上の価値があります。",
    "品質が悪くて残念です。", "まったく役に立ちません。",
    "すぐに壊れてしまいました。", "二度と購入しません。",
    "説明が不十分で困りました。", "価格に見合わない商品です。",
]
labels = [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]

train_texts, test_texts, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels
)

In [6]:
# Ruri v3では分類用文章に "トピック: " を付ける
X_train = encoder.encode(["トピック: " + x for x in train_texts], normalize_embeddings=True)
X_test = encoder.encode(["トピック: " + x for x in test_texts], normalize_embeddings=True)
print(X_train.shape)  # (学習件数, 768)

(9, 256)


In [7]:
classifier = LogisticRegression(max_iter=1000, random_state=42)
classifier.fit(X_train, y_train)

pred = classifier.predict(X_test)
print("accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, zero_division=0))

accuracy: 0.3333333333333333
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.33      1.00      0.50         1

    accuracy                           0.33         3
   macro avg       0.17      0.50      0.25         3
weighted avg       0.11      0.33      0.17         3



In [8]:
def predict(text):
    embedding = encoder.encode(["トピック: " + text], normalize_embeddings=True)
    label = int(classifier.predict(embedding)[0])
    probability = float(classifier.predict_proba(embedding)[0, label])
    return {"label": label, "probability": probability}

predict("操作が簡単で、とても気に入りました。")

{'label': 1, 'probability': 0.5844589182083102}